In [6]:
from pymatgen.ext.matproj import MPRester
from pymatgen.io.phonopy import get_phonopy_structure
from phonopy import Phonopy
from pymatgen.symmetry.bandstructure import HighSymmKpath
import numpy as np
import plotly.graph_objects as go
import itertools

API_KEY = "zA1YwwFoS7C8RQ3ZpVn2lK2ClRoSNTFM"
mpr = MPRester(API_KEY)
structure = mpr.get_structure_by_material_id("mp-27869")
ph_bs = mpr.get_phonon_bandstructure_by_material_id("mp-27869")

#Def de la fonction d'affichage de la zone de Brillouin
def latex_fix(label):
    replace = { "\Gamma": "Γ" }
    if label in replace:
        label = replace[label]
    return label

# Plotting of the Brillouin zone
def go_points(points, size=4, color="black", labels=None):
    mode = "markers" if labels is None else "markers+text"
    if labels is not None:
        for il in range(len(labels)):
            labels[il] = latex_fix(labels[il])
    return go.Scatter3d(
        x=[v[0] for v in points],
        y=[v[1] for v in points],
        z=[v[2] for v in points],
        marker=dict(size=size, color=color),
        mode=mode,
        text=labels,
        textfont_color=color,
        showlegend=False
    )

def go_line(v1, v2, color="black", width=2, mode="lines", text=""):
    return go.Scatter3d(
        mode=mode,
        x=[v1[0], v2[0]],
        y=[v1[1], v2[1]],
        z=[v1[2], v2[2]],
        line=dict(color=color),
        text=text,
        showlegend=False
    )

def plot_brillouin_zone(struc, fig=None):
    bz_lattice = struc.lattice.reciprocal_lattice
    if fig is None:
        fig = go.Figure()

#   Plot the three lattice vectors
    vertex1 = bz_lattice.get_cartesian_coords([0.0, 0.0, 0.0])
    vertex2 = bz_lattice.get_cartesian_coords([1.0, 0.0, 0.0])
    fig.add_trace(go_line(vertex1, vertex2, color="green", mode="lines+text", text=["","a"]))
    vertex2 = bz_lattice.get_cartesian_coords([0.0, 1.0, 0.0])
    fig.add_trace(go_line(vertex1, vertex2, color="green", mode="lines+text", text=["","b"]))
    vertex2 = bz_lattice.get_cartesian_coords([0.0, 0.0, 1.0])
    fig.add_trace(go_line(vertex1, vertex2, color="green", mode="lines+text", text=["","c"]))
    
#   Plot the Wigner-Seitz cell
    bz = bz_lattice.get_wigner_seitz_cell()
    for iface in range(len(bz)): # pylint: disable=C0200
        for line in itertools.combinations(bz[iface], 2):
            for jface in range(len(bz)):
                if (iface < jface
                    and any(np.all(line[0] == x) for x in bz[jface])
                    and any(np.all(line[1] == x) for x in bz[jface])):
                    fig.add_trace(go_line(line[0], line[1]))

#   Plot the path in the Brillouin zone
    kpath = HighSymmKpath(struc)
    for line in [[kpath.kpath["kpoints"][k] for k in p] for p in kpath.kpath["path"]]:
        for k in range(1, len(line)):
            vertex1 = line[k - 1]
            vertex2 = line[k]
            vertex1 = bz_lattice.get_cartesian_coords(vertex1)
            vertex2 = bz_lattice.get_cartesian_coords(vertex2)
            fig.add_trace(go_line(vertex1, vertex2, color="red"))

    # Points
    labels = kpath.kpath["kpoints"]
    vecs = []
    for point in labels.values():
        vecs.append(bz_lattice.get_cartesian_coords(point))

    fig.add_trace(go_points(vecs, color="red", labels=list(labels.keys())))

    fig.update_layout(
        scene=dict(
            xaxis=dict(visible=False, range=[-1.15, 1.15]),
            yaxis=dict(visible=False, range=[-1.15, 1.15]),
            zaxis=dict(visible=False, range=[-1.15, 1.15])
        )
    )
    return fig

fig = plot_brillouin_zone(structure)
fig.show()

#Calcul de la vitesse du son
kpath = HighSymmKpath(structure)
kpoints = kpath.kpath["kpoints"]

Γ = np.array(kpoints["Γ"])
X = np.array(kpoints["X"])

rec_latt = structure.lattice.reciprocal_lattice
kΓ = rec_latt.get_cartesian_coords(Γ)
kX = rec_latt.get_cartesian_coords(X)
delta_k = np.linalg.norm(kX - kΓ)

freqs = ph_bs.bands[0][:2]
delta_f = freqs[1] - freqs[0]

v_son = (delta_f / delta_k) * 1e2

print(f"Vitesse du son Γ–X (branche acoustique 1) : {v_son:.2f} m/s")







Retrieving MaterialsDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieving PhononBSDOSDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

KeyError: 'Γ'